# LeetCode 3630 — Partition Array for Maximum XOR and AND bằng Python

- Link: https://leetcode.com/problems/partition-array-for-maximum-xor-and-and/
- Độ khó: **Hard**
- Chủ đề: Duyệt tập con (`n ≤ 19`), **cơ sở tuyến tính XOR**, đẳng thức `a + b = (a ⊕ b) + 2(a & b)`, DP trên mặt nạ, **cắt nhánh bằng cận trên**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maximizeXorAndXor(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Với `n ≤ 19`, duyệt mọi tập `B` (`2^19 ≈ 5·10^5`); phần còn lại `R` chia thành `A`, `C`.
2. Đẳng thức then chốt: với `a = XOR(A)`, `c = XOR(C)`, `a ⊕ c = T := XOR(R)` cố định ⇒ `a + c = a + (a ⊕ T) = T + 2·(a & ~T)`.
3. ⇒ cần **XOR lớn nhất** của tập con của `{x & ~T : x ∈ R}` ⇒ **cơ sở tuyến tính** (như bài 3681).
4. Tính `AND`, `XOR`, `OR` của mọi mặt nạ bằng DP "bỏ bit thấp nhất" trong `O(2^n)`.
5. Python quá chậm nếu dựng cơ sở cho cả `2^19` mặt nạ ⇒ sắp mặt nạ theo **cận trên** `AND(B) + T + 2·(OR(R) & ~T)` giảm dần và dừng sớm.

## 2. Đề bài & ví dụ

Chia `nums` thành ba dãy con `A`, `B`, `C` (có thể rỗng, mỗi phần tử thuộc đúng một dãy). Tối đa hoá `XOR(A) + AND(B) + XOR(C)` (dãy rỗng cho giá trị 0).

**Ví dụ 1:** `[2,3]` → `5`.

**Ví dụ 2:** `[1,3,2]` → `6`.

**Ví dụ 3:** `[2,3,6,7]` → `15`.

**Ràng buộc:** `n ≤ 19`, `nums[i] ≤ 10^9`.

In [1]:
from typing import List
from itertools import product
import random, time

examples = [([2, 3], 5), ([1, 3, 2], 6), ([2, 3, 6, 7], 15)]
for a, exp in examples:
    print(a, "-> kỳ vọng", exp)

[2, 3] -> kỳ vọng 5
[1, 3, 2] -> kỳ vọng 6
[2, 3, 6, 7] -> kỳ vọng 15


## 3. Kiến thức nền

### 3.1. `a + b = (a ⊕ b) + 2(a & b)`

Cộng nhị phân: XOR là tổng không nhớ, AND là vị trí sinh nhớ (nhớ dịch trái 1 bit). Với `b = a ⊕ T`: `a & (a ⊕ T) = a & ~T` ⇒ `a + (a ⊕ T) = T + 2(a & ~T)`.

In [2]:
rng0 = random.Random(0)
for _ in range(5):
    a, T = rng0.randint(0, 1000), rng0.randint(0, 1000)
    assert a + (a ^ T) == T + 2 * (a & ~T)
print("đẳng thức đúng; ví dụ a=5, T=6:", 5 + (5 ^ 6), "=", 6 + 2 * (5 & ~6))

đẳng thức đúng; ví dụ a=5, T=6: 8 = 8


### 3.2. Cơ sở XOR và XOR lớn nhất

Chèn từng số, khử bằng các vectơ cơ sở (`x = min(x, x ⊕ b)` là mẹo gọn khi cơ sở được giữ ở dạng "mỗi vectơ có bit cao nhất riêng"); XOR lớn nhất = tham lam từ vectơ có bit cao nhất xuống.

In [3]:
def max_xor(values):
    basis = []
    for x in values:
        for b in basis: x = min(x, x ^ b)
        if x: basis.append(x)
    best = 0
    for b in sorted(basis, reverse=True): best = max(best, best ^ b)
    return best

print(max_xor([3, 10, 5, 25, 2, 8]))

31


### 3.3. DP trên mặt nạ

`AND[m] = AND[m bỏ bit thấp nhất] & nums[bit đó]` (và tương tự cho `XOR`, `OR`) — tính cả `2^n` mặt nạ trong `O(2^n)`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Cố định `B` (mặt nạ `m`), `R = phần bù`, `T = XOR(R)`. Mọi cách chia `R` thành `A ∪ C` cho `a ⊕ c = T`, và `a` chạy trên **toàn bộ không gian XOR** của `R` (mỗi `a` đạt được bởi một tập con `A`). Vậy:

`giá trị(m) = AND(B) + T + 2·max_{a ∈ span(R)} (a & ~T) = AND(B) + T + 2·maxXOR({x & ~T : x ∈ R})`

(bước cuối vì `(·) & ~T` là ánh xạ tuyến tính trên GF(2)).

Ví dụ 3, `B = {2, 3}`: `AND = 2`, `R = {6, 7}`, `T = 1`; `{6 & ~1, 7 & ~1} = {6, 6}` ⇒ max XOR `6` ⇒ `2 + 1 + 12 = 15`. ✔

## 5. Cách 1 — Brute Force: `3^n` cách gán

In [4]:
def brute(nums):
    best = 0
    for assign in product(range(3), repeat=len(nums)):
        xa = xc = 0; b = None
        for v, g in zip(nums, assign):
            if g == 0: xa ^= v
            elif g == 2: xc ^= v
            else: b = v if b is None else b & v
        best = max(best, xa + xc + (b or 0))
    return best

for a, exp in examples:
    print(brute(a), "kỳ vọng", exp)

5 kỳ vọng 5
6 kỳ vọng 6
15 kỳ vọng 15


**Nút thắt:** `3^19 ≈ 1,16·10^9`.

## 6. Ý tưởng thuật toán / bước đột phá

`2^n` mặt nạ × dựng cơ sở `O(n · 30)` ⇒ `≈ 3·10^8` phép — ổn với C++ nhưng chậm trong Python. **Cắt nhánh:** vì `a & ~T ≤ OR(R) & ~T`, ta có cận trên `UB(m) = AND(B) + T + 2·(OR(R) & ~T)`. Sắp mặt nạ theo `UB` giảm dần, tính giá trị thật cho từng mặt nạ, và **dừng** ngay khi `UB ≤ best` (mọi mặt nạ sau đều không thể tốt hơn). Trên thực tế chỉ vài mặt nạ đầu cần dựng cơ sở.

## 7. Các cách tiếp theo

- **Cách 2:** duyệt mọi `B` + cơ sở tuyến tính (không cắt nhánh) — đúng, `O(2^n · n · 30)`.
- **Cách 3:** như Cách 2 + DP mặt nạ + cắt nhánh theo cận trên (lời giải nộp).
- **Cách 4:** duyệt `B` nhưng thay cơ sở bằng **liệt kê không gian XOR** của `R` khi `|R|` nhỏ — minh hoạ vì sao cơ sở cần thiết.

### 7.1. Cách 2 — Duyệt `B` + cơ sở

In [5]:
def all_masks_basis(nums):
    n = len(nums); best = 0
    for m in range(1 << n):
        B = [nums[i] for i in range(n) if m >> i & 1]; R = [nums[i] for i in range(n) if not m >> i & 1]
        andB = 0
        if B:
            andB = B[0]
            for v in B[1:]: andB &= v
        T = 0
        for v in R: T ^= v
        best = max(best, andB + T + 2 * max_xor([x & ~T for x in R]))
    return best

for a, exp in examples:
    print(all_masks_basis(a), "kỳ vọng", exp)

5 kỳ vọng 5
6 kỳ vọng 6
15 kỳ vọng 15


### 7.2. Cách 3 — DP mặt nạ + cắt nhánh (lời giải chính)

In [6]:
def pruned(nums):
    n = len(nums); N = 1 << n; full = N - 1
    AND = [0] * N; XOR = [0] * N; OR = [0] * N; AND[0] = -1
    for m in range(1, N):
        low = m & -m; i = low.bit_length() - 1; p = m ^ low
        AND[m] = AND[p] & nums[i]; XOR[m] = XOR[p] ^ nums[i]; OR[m] = OR[p] | nums[i]
    AND[0] = 0
    ub = [AND[m] + XOR[full ^ m] + 2 * (OR[full ^ m] & ~XOR[full ^ m]) for m in range(N)]
    best = 0
    for m in sorted(range(N), key=ub.__getitem__, reverse=True):
        if ub[m] <= best: break
        r = full ^ m; T = XOR[r]
        vals = [nums[i] & ~T for i in range(n) if r >> i & 1]
        best = max(best, AND[m] + T + 2 * max_xor(vals))
    return best

for a, exp in examples:
    print(pruned(a), "kỳ vọng", exp)

5 kỳ vọng 5
6 kỳ vọng 6
15 kỳ vọng 15


### 7.3. Cách 4 — Liệt kê không gian XOR

In [7]:
def enumerate_span(nums):
    n = len(nums); best = 0
    for m in range(1 << n):
        B = [nums[i] for i in range(n) if m >> i & 1]; R = [nums[i] for i in range(n) if not m >> i & 1]
        andB = 0
        if B:
            andB = B[0]
            for v in B[1:]: andB &= v
        T = 0
        for v in R: T ^= v
        span = {0}
        for x in R: span |= {s ^ x for s in span}
        best = max(best, andB + max(a + (a ^ T) for a in span))
    return best

for a, exp in examples:
    print(enumerate_span(a), "kỳ vọng", exp)

5 kỳ vọng 5
6 kỳ vọng 6
15 kỳ vọng 15


**Nhận xét:** không gian có tới `2^{|R|}` phần tử ⇒ tổng `≈ 3^n` — chính là vét cạn trá hình; cơ sở tuyến tính nén nó còn `O(|R| · 30)`.

## 8. TRACE từng bước

In [8]:
def trace(nums, top=6):
    n = len(nums); N = 1 << n; full = N - 1
    rows = []
    for m in range(N):
        B = [nums[i] for i in range(n) if m >> i & 1]; R = [nums[i] for i in range(n) if not m >> i & 1]
        andB = 0
        if B:
            andB = B[0]
            for v in B[1:]: andB &= v
        T = 0; orR = 0
        for v in R: T ^= v; orR |= v
        ub = andB + T + 2 * (orR & ~T); val = andB + T + 2 * max_xor([x & ~T for x in R])
        rows.append((ub, val, B, R, T))
    rows.sort(key=lambda r: -r[0]); best = 0
    for ub, val, B, R, T in rows[:top]:
        stop = ub <= best
        print(f"B={B}, R={R}, T={T}: cận trên {ub}, giá trị thật {val}" + ("  <- dừng (UB <= best)" if stop else ""))
        if stop: break
        best = max(best, val)
    print("đáp án:", best)

trace([2, 3, 6, 7])

B=[2, 3], R=[6, 7], T=1: cận trên 15, giá trị thật 15
B=[], R=[2, 3, 6, 7], T=0: cận trên 14, giá trị thật 14  <- dừng (UB <= best)
đáp án: 15


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def maximizeXorAndXor(self, nums: List[int]) -> int:
        n = len(nums)
        N = 1 << n
        full = N - 1
        AND = [0] * N
        XOR = [0] * N
        OR = [0] * N
        AND[0] = -1
        for m in range(1, N):
            low = m & -m
            i = low.bit_length() - 1
            p = m ^ low
            AND[m] = AND[p] & nums[i]
            XOR[m] = XOR[p] ^ nums[i]
            OR[m] = OR[p] | nums[i]
        AND[0] = 0
        ub = [AND[m] + XOR[full ^ m] + 2 * (OR[full ^ m] & ~XOR[full ^ m]) for m in range(N)]
        best = 0
        for m in sorted(range(N), key=ub.__getitem__, reverse=True):
            if ub[m] <= best:
                break
            r = full ^ m
            T = XOR[r]
            nt = ~T
            basis = []
            for i in range(n):
                if r >> i & 1:
                    x = nums[i] & nt
                    for b in basis:
                        x = min(x, x ^ b)
                    if x:
                        basis.append(x)
            mx = 0
            for b in sorted(basis, reverse=True):
                if mx ^ b > mx:
                    mx ^= b
            val = AND[m] + T + 2 * mx
            if val > best:
                best = val
        return best

sol = Solution()
for a, exp in examples:
    got = sol.maximizeXorAndXor(a)
    print(a, "->", got, "✔" if got == exp else "✘")

[2, 3] -> 5 ✔
[1, 3, 2] -> 6 ✔
[2, 3, 6, 7] -> 15 ✔


## 10. Vì sao thuật toán đúng

1. **Tách theo `B`:** đáp án `= max_B [AND(B) + max_{A ⊔ C = R} (XOR(A) + XOR(C))]`.
2. **Phần `A, C`:** `XOR(A) ⊕ XOR(C) = T`. Khi `A` chạy trên mọi tập con của `R`, `a = XOR(A)` chạy trên đúng `span(R)`, và `c = a ⊕ T`. Theo 3.1, `a + c = T + 2(a & ~T)`.
3. **Tuyến tính:** `a ↦ a & ~T` là ánh xạ tuyến tính trên GF(2) ⇒ ảnh của `span(R)` là `span({x & ~T})` ⇒ max cần tìm là XOR lớn nhất của cơ sở đó.
4. **Cắt nhánh đúng:** `a & ~T ⊆ OR(R) & ~T` (theo bit) ⇒ `a & ~T ≤ OR(R) & ~T` ⇒ `val(m) ≤ UB(m)`. Duyệt theo `UB` giảm dần: khi `UB(m) ≤ best`, mọi mặt nạ sau có `UB ≤ best` ⇒ không cải thiện được ⇒ dừng an toàn.
5. `B = ∅` có `AND = 0` theo quy ước — mặt nạ `0` xử lý riêng (`AND[0] = 0`).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(3^n · n)` | `O(n)` |
| 2. Duyệt `B` + cơ sở | `O(2^n · n · 30)` | `O(n)` |
| 3. DP mặt nạ + cắt nhánh | `O(2^n log 2^n)` (sắp) + số mặt nạ không bị cắt × `O(n·30)` | `O(2^n)` |
| 4. Liệt kê không gian | `O(3^n)` | `O(2^n)` |

Thực nghiệm: với `n = 19`, Cách 3 chỉ dựng cơ sở cho một số rất nhỏ mặt nạ — tổng ~0,5 giây.

## 12. Bộ test

In [10]:
METHODS = [brute, all_masks_basis, pruned, enumerate_span, lambda a: Solution().maximizeXorAndXor(a)]
for a, exp in examples:
    assert all(f(a) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 ca biên
assert Solution().maximizeXorAndXor([7]) == 7
assert Solution().maximizeXorAndXor([5, 5]) == 10
assert Solution().maximizeXorAndXor([1, 1, 1]) == brute([1, 1, 1])
print("Ca biên: OK")

Ca biên: OK


In [12]:
# 12.3 ngẫu nhiên
rng = random.Random(3630)
for _ in range(400):
    a = [rng.randint(1, 31) for _ in range(rng.randint(1, 7))]
    b = brute(a)
    assert all(f(a) == b for f in METHODS[1:]), a
for _ in range(30):
    a = [rng.randint(1, 10**9) for _ in range(rng.randint(8, 12))]
    assert pruned(a) == all_masks_basis(a)
print("430 test ngẫu nhiên: OK")

430 test ngẫu nhiên: OK


In [13]:
# 12.4 thời gian với n = 19 trên nhiều kiểu dữ liệu
pats = {"ngẫu nhiên": [rng.randint(1, 10**9) for _ in range(19)], "bằng nhau": [10**9] * 19,
        "lũy thừa 2": [1 << i for i in range(19)], "liên tiếp": list(range(1, 20)),
        "gần 2^30": [(1 << 30) - 1 - (1 << i) for i in range(19)]}
for name, a in pats.items():
    t0 = time.time(); x = Solution().maximizeXorAndXor(a); t1 = time.time()
    print(f"{name}: {x} | {t1 - t0:.2f}s")

ngẫu nhiên: 2935867022 | 0.44s


bằng nhau: 3000000000 | 0.28s
lũy thừa 2: 524287 | 0.23s


liên tiếp: 79 | 0.23s


gần 2^30: 3220701182 | 0.33s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cho `A` và `C` gộp lại (bỏ qua việc tách thành hai XOR)

Chỉ tính `XOR(R)` thay vì `max XOR(A) + XOR(C)`.

In [14]:
def wrong_merge(nums):
    n = len(nums); best = 0
    for m in range(1 << n):
        B = [nums[i] for i in range(n) if m >> i & 1]; R = [nums[i] for i in range(n) if not m >> i & 1]
        andB = 0
        if B:
            andB = B[0]
            for v in B[1:]: andB &= v
        T = 0
        for v in R: T ^= v
        best = max(best, andB + T)                                        # SAI
    return best

rng = random.Random(8)
while True:
    a = [rng.randint(1, 15) for _ in range(rng.randint(2, 5))]
    if wrong_merge(a) != brute(a): break
print("phản ví dụ:", a, "| sai:", wrong_merge(a), "| đúng:", brute(a))

phản ví dụ: [6, 7, 3] | sai: 12 | đúng: 16


### Lỗi 2 — Lấy `max XOR(span(R))` rồi cộng với `T` (sai công thức)

In [15]:
def wrong_formula(nums):
    n = len(nums); best = 0
    for m in range(1 << n):
        B = [nums[i] for i in range(n) if m >> i & 1]; R = [nums[i] for i in range(n) if not m >> i & 1]
        andB = 0
        if B:
            andB = B[0]
            for v in B[1:]: andB &= v
        T = 0
        for v in R: T ^= v
        a = max_xor(R); best = max(best, andB + a + (a ^ T))               # SAI: a tối đa không làm a + (a^T) tối đa
    return best

rng = random.Random(9)
while True:
    a = [rng.randint(1, 15) for _ in range(rng.randint(2, 6))]
    if wrong_formula(a) != brute(a): break
print("phản ví dụ:", a, "| sai:", wrong_formula(a), "| đúng:", brute(a))

phản ví dụ: [1, 1, 13, 3] | sai: 16 | đúng: 17


### Lỗi 3 — Khởi tạo `AND` của tập rỗng bằng "toàn bit 1" rồi quên đặt lại

Trong DP mặt nạ, `AND[0]` phải là phần tử trung hoà (toàn bit 1) để `AND[{i}] = nums[i]`. Nếu quên đặt lại `AND[0] = 0` trước khi tính giá trị, tập `B = ∅` được "thưởng" `2^{30} − 1` — đáp án bị thổi phồng.

In [16]:
def wrong_empty_and(nums):
    n = len(nums); N = 1 << n; full = N - 1; AND = [0] * N; XOR = [0] * N; AND[0] = (1 << 30) - 1
    for m in range(1, N):
        low = m & -m; i = low.bit_length() - 1; p = m ^ low
        AND[m] = AND[p] & nums[i]; XOR[m] = XOR[p] ^ nums[i]
    best = 0                                                               # SAI: quên đặt AND[0] = 0
    for m in range(N):
        r = full ^ m; T = XOR[r]
        best = max(best, AND[m] + T + 2 * max_xor([nums[i] & ~T for i in range(n) if r >> i & 1]))
    return best

print("sai:", wrong_empty_and([2, 3]), "| đúng:", brute([2, 3]))
assert wrong_empty_and([2, 3]) != brute([2, 3])

sai: 1073741828 | đúng: 5


### Lỗi 4 — Cắt nhánh khi `UB < best` nhưng **không** sắp theo `UB`

Dừng sớm chỉ hợp lệ khi các mặt nạ được duyệt theo cận trên **giảm dần**; duyệt theo thứ tự mặt nạ tự nhiên rồi `break` có thể bỏ sót.

In [17]:
def wrong_unsorted_break(nums):
    n = len(nums); N = 1 << n; full = N - 1
    AND = [0] * N; XOR = [0] * N; OR = [0] * N; AND[0] = -1
    for m in range(1, N):
        low = m & -m; i = low.bit_length() - 1; p = m ^ low
        AND[m] = AND[p] & nums[i]; XOR[m] = XOR[p] ^ nums[i]; OR[m] = OR[p] | nums[i]
    AND[0] = 0; best = 0
    for m in range(N):                                                     # SAI: không sắp
        r = full ^ m; T = XOR[r]
        if AND[m] + T + 2 * (OR[r] & ~T) <= best: break
        best = max(best, AND[m] + T + 2 * max_xor([nums[i] & ~T for i in range(n) if r >> i & 1]))
    return best

rng = random.Random(10)
while True:
    a = [rng.randint(1, 31) for _ in range(rng.randint(2, 6))]
    if wrong_unsorted_break(a) != brute(a): break
print("phản ví dụ:", a, "| sai:", wrong_unsorted_break(a), "| đúng:", brute(a))

phản ví dụ: [2, 14, 16, 19, 1, 7] | sai: 53 | đúng: 55


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- `n ≤ 20` ⇒ duyệt mặt nạ cho phần "khó" (ở đây là `B`), giải phần còn lại bằng cấu trúc đẹp.
- "Tổng hai XOR khi XOR của chúng cố định" ⇒ `T + 2(a & ~T)` ⇒ cơ sở trên `x & ~T`.
- AND/OR/XOR của mọi tập con ⇒ DP "bỏ bit thấp nhất".
- Python + `2^19` vòng nặng ⇒ tìm **cận trên** rẻ để cắt nhánh (branch and bound).

## 15. Biến thể của bài

1. **Hai phần `A`, `C` (không có `B`)**: `max_a a + (a ⊕ T)` — chỉ một lần dựng cơ sở.
2. **Tối đa hoá `XOR(A) · XOR(C)`**: không còn tuyến tính — khó hơn nhiều.
3. **`OR(B)` thay `AND(B)`**: giữ nguyên khung, đổi DP.
4. **`k` nhóm XOR**: tổng `k` XOR có cùng XOR chung… với `k ≥ 3` không còn công thức gọn.

In [18]:
# Biến thể 1 và 3
def two_parts(nums):
    T = 0
    for v in nums: T ^= v
    return T + 2 * max_xor([x & ~T for x in nums])

def or_variant(nums):
    n = len(nums); best = 0
    for m in range(1 << n):
        orB = 0; T = 0; R = []
        for i in range(n):
            if m >> i & 1: orB |= nums[i]
            else: T ^= nums[i]; R.append(nums[i])
        best = max(best, orB + T + 2 * max_xor([x & ~T for x in R]))
    return best

def brute_or(nums):
    best = 0
    for assign in product(range(3), repeat=len(nums)):
        xa = xc = ob = 0
        for v, g in zip(nums, assign):
            if g == 0: xa ^= v
            elif g == 2: xc ^= v
            else: ob |= v
        best = max(best, xa + xc + ob)
    return best

rng = random.Random(15)
for _ in range(200):
    a = [rng.randint(1, 31) for _ in range(rng.randint(1, 6))]
    bt = 0
    for g in product(range(2), repeat=len(a)):
        xa = xc = 0
        for v, side in zip(a, g):
            if side: xc ^= v
            else: xa ^= v
        bt = max(bt, xa + xc)
    assert two_parts(a) == bt
    assert or_variant(a) == brute_or(a)
print("Biến thể 1, 3 OK")

Biến thể 1, 3 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | `3^n` | `O(3^n n)` | Kiểm chứng |
| 2. `B` + cơ sở | Đẳng thức + tuyến tính | `O(2^n · n · 30)` | C++/Java |
| 3. + cắt nhánh | Cận trên `OR(R) & ~T` | rất nhanh thực tế | **Python** |
| 4. Liệt kê không gian | Không dùng cơ sở | `O(3^n)` | Minh hoạ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `maximizeXorAndXor([8])`.

**Bài 2 (tính tay).** `maximizeXorAndXor([4, 4, 4])`.

**Bài 3 (điền TODO).** Viết `best_partition(nums)` trả về luôn ba danh sách `A, B, C` đạt tối ưu (cho `n ≤ 12`).

**Bài 4 (tự viết lại).** Chứng minh bằng thực nghiệm rằng thứ tự duyệt theo `UB` thực sự chỉ dựng cơ sở cho ít mặt nạ: đếm số mặt nạ được đánh giá với `n = 19`.

**Bài 5 (mini challenge).** Thay `AND(B)` bằng `max(B)` (phần tử lớn nhất của `B`, 0 nếu rỗng) và giải.

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def best_partition(nums):
    # TODO: với B tối ưu, dựng A bằng cách biểu diễn a tối ưu qua cơ sở (lưu tổ hợp chỉ số)
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def max_variant(nums):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Một phần tử: cho vào `A` (8) hoặc `B` (8). (2) Không thể hơn. (3) `8`.

**Bài 2.** (1) `B = {4,4,4}` cho `4`. (2) `A = {4}, B = {4}, C = {4}` cho `12`. (3) `12`.

**Bài 3.** (1) Dựng cơ sở kèm "tập chỉ số tạo ra vectơ". (2) Khi tham lam chọn vectơ, XOR tập chỉ số. (3) `A` = tập chỉ số thu được, `C = R \ A`.

**Bài 4.** (1) Đếm số lần vào thân vòng lặp. (2) So với `2^19`. (3) In tỉ lệ.

**Bài 5.** (1) Khung giữ nguyên: duyệt mọi `B`, phần `A, C` vẫn là `T + 2·maxXOR({x & ~T})`. (2) Chỉ đổi `AND(B)` thành `max(B)`. (3) Cẩn thận: **không** thể chỉ xét `B` một phần tử — đưa thêm phần tử nhỏ vào `B` làm thay đổi `T` của phần còn lại và có thể tăng tổng (ví dụ `[9, 24, 12, 26, 23]`).

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
assert Solution().maximizeXorAndXor([8]) == 8
assert Solution().maximizeXorAndXor([4, 4, 4]) == 12 == brute([4, 4, 4])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [22]:
# Bài 3, 4
def best_partition_sol(nums):
    n = len(nums); best = (-1, None)
    for m in range(1 << n):
        idxR = [i for i in range(n) if not m >> i & 1]; T = 0
        for i in idxR: T ^= nums[i]
        andB = 0; first = True
        for i in range(n):
            if m >> i & 1: andB = nums[i] if first else andB & nums[i]; first = False
        basis = []                                            # (vectơ, mặt nạ chỉ số)
        for i in idxR:
            x, msk = nums[i] & ~T, 1 << i
            for b, bm in basis:
                if x ^ b < x: x ^= b; msk ^= bm
            if x: basis.append((x, msk))
        a = 0; amsk = 0
        for b, bm in sorted(basis, reverse=True):
            if a ^ b > a: a ^= b; amsk ^= bm
        val = andB + T + 2 * a
        if val > best[0]:
            A = [nums[i] for i in range(n) if amsk >> i & 1]; Bl = [nums[i] for i in range(n) if m >> i & 1]
            C = [nums[i] for i in idxR if not amsk >> i & 1]; best = (val, (A, Bl, C))
    return best

rng = random.Random(4)
for _ in range(100):
    a = [rng.randint(1, 31) for _ in range(rng.randint(1, 6))]
    val, (A, Bl, C) = best_partition_sol(a)
    xa = xc = 0
    for v in A: xa ^= v
    for v in C: xc ^= v
    ab = 0
    if Bl:
        ab = Bl[0]
        for v in Bl[1:]: ab &= v
    assert val == brute(a) == xa + ab + xc and sorted(A + Bl + C) == sorted(a)
print("Bài 3 OK; ví dụ 3:", best_partition_sol([2, 3, 6, 7]))

def count_evaluated(nums):
    n = len(nums); N = 1 << n; full = N - 1
    AND = [0] * N; XOR = [0] * N; OR = [0] * N; AND[0] = -1
    for m in range(1, N):
        low = m & -m; i = low.bit_length() - 1; p = m ^ low
        AND[m] = AND[p] & nums[i]; XOR[m] = XOR[p] ^ nums[i]; OR[m] = OR[p] | nums[i]
    AND[0] = 0; ub = [AND[m] + XOR[full ^ m] + 2 * (OR[full ^ m] & ~XOR[full ^ m]) for m in range(N)]
    best = 0; cnt = 0
    for m in sorted(range(N), key=ub.__getitem__, reverse=True):
        if ub[m] <= best: break
        cnt += 1; r = full ^ m; T = XOR[r]
        best = max(best, AND[m] + T + 2 * max_xor([nums[i] & ~T for i in range(n) if r >> i & 1]))
    return cnt, N

print("Bài 4: số mặt nạ phải dựng cơ sở / tổng:", count_evaluated([rng.randint(1, 10**9) for _ in range(19)]))

Bài 3 OK; ví dụ 3: (15, ([6], [2, 3], [7]))


Bài 4: số mặt nạ phải dựng cơ sở / tổng: (1, 524288)


In [23]:
# Bài 5: max(B) thay cho AND(B)
def max_variant_sol(nums):
    n = len(nums); best = 0
    for m in range(1 << n):                                         # vẫn phải duyệt mọi B: thêm phần tử nhỏ vào B làm đổi T
        mb = 0; T = 0; R = []
        for i in range(n):
            if m >> i & 1: mb = max(mb, nums[i])
            else: T ^= nums[i]; R.append(nums[i])
        best = max(best, mb + T + 2 * max_xor([x & ~T for x in R]))
    return best

def brute_max(nums):
    best = 0
    for assign in product(range(3), repeat=len(nums)):
        xa = xc = mb = 0
        for v, g in zip(nums, assign):
            if g == 0: xa ^= v
            elif g == 2: xc ^= v
            else: mb = max(mb, v)
        best = max(best, xa + xc + mb)
    return best

rng = random.Random(5)
for _ in range(300):
    a = [rng.randint(1, 31) for _ in range(rng.randint(1, 6))]
    assert max_variant_sol(a) == brute_max(a), a
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
DP mặt nạ: AND[m], XOR[m], OR[m] (AND[0] = 0 sau khi dựng)
UB(m) = AND[m] + T + 2*(OR[R] & ~T),  T = XOR[R], R = full ^ m
for m theo UB giảm dần:
    if UB(m) <= best: break
    mx = maxXOR({x & ~T : x ∈ R})
    best = max(best, AND[m] + T + 2*mx)
```

**Checklist:**

- [ ] Dùng `a + (a ⊕ T) = T + 2(a & ~T)`?
- [ ] Cơ sở trên `x & ~T` (không phải `x`)?
- [ ] `AND(∅) = 0`?
- [ ] Duyệt theo cận trên **giảm dần** trước khi `break`?
- [ ] Đã so với vét cạn `3^n`?